# Load a pre-trained model

### A. Import the untrained version of the model

In [ ]:
import jax
import jax.numpy as jnp
import flax.nnx as nnx

from src.model import NanoLLM
from src.inference import generate_text
from src.config import ModelConfig, model_config, TokenizerConfig

In [ ]:
config = model_config
print("Model config:")
print(f"maxlen: {config.maxlen}") # inference constraint
print(f"num_transformer_blocks: {config.num_transformer_blocks}") # model depth
print(f"feed_forward_dim: {config.feed_forward_dim}") # model capacity
print(f"vocab_size: {config.vocab_size}") # validation check
print(f"embed_dim: {config.embed_dim}")
print(f"num_heads: {config.num_heads}")

In [ ]:
model = NanoLLM()
model

### B. Capture and inspect initial structure of the imported model

In [ ]:
# Option 1: Norm (condensed option to save memory)
# This stores one number per layer, potentially hundreds of numbers total
initial_norm = jax.tree_util.tree_map(
    lambda x: jnp.linalg.norm(x), nnx.state(model)
)

initial_norm

In [ ]:
# Option 2: State (verbose option) 
# This stores every parameter - a complete snapshot of the model (potentially millions of numbers)
initial_state = nnx.state(model)

### C. Load the most recently saved checkpoint

In [ ]:
import orbax
from orbax import checkpoint
from src.config import CHECKPOINTS_DIR

# to deal with a model that was trained on many machines (likely GPUs), but will be loaded to one machine (our CPU)
from jax.sharding import SingleDeviceSharding

In [ ]:
# 1. Locate and define current machine on which you will run the model
cpu_device = jax.devices('cpu')[0]
cpu_sharding = SingleDeviceSharding(cpu_device)

In [ ]:
# 2. Add capability to retrieve the checkpoint onto as single CPU
restore_args = jax.tree_util.tree_map(

    # for each array of parameters, restore that array to the single CPU device 
    lambda _: checkpoint.ArrayRestoreArgs(sharding=cpu_sharding),

    # returns the shape of the model
    nnx.state(model)
)

In [ ]:
# 3. Load the last checkpoint

checkpoint_path = CHECKPOINTS_DIR / "nano_checkpoint.orbax"
checkpointer = orbax.checkpoint.PyTreeCheckpointer()

### D. Update the untrained model with the parameters from the checkpoint

In [ ]:
# Use the below to update a model with the checkpoints above
restored_state = checkpointer.restore(
    checkpoint_path,
    item=nnx.state(model),
    restore_args=restore_args)

In [ ]:
# Update the model with the loaded checkpoints
nnx.update(model,restored_state)

In [ ]:
# Capture the resulting state of the model

# Option 1: Norm (condensed option to save memory)
resulting_norm = jax.tree_util.tree_map(
    lambda x: jnp.linalg.norm(x), nnx.state(model)
)

# Option 2: State (verbose option) 
resulting_state = nnx.state(model)

### E. Compare the model before and after

In [ ]:
def analyze_state_changes(state1, state2):
    """Analyze differences between two model states"""
    
    # Calculate differences for each parameter array
    diffs = jax.tree_util.tree_map(
        lambda x, y: jnp.abs(x - y),
        state1, state2
    )
    
    # Flatten to get all individual statistics
    all_diffs = jax.tree_util.tree_leaves(diffs)

    # Count parameters that changed
    total_params = sum(x.size for x in all_diffs)
    changed_params = sum((x > 1e-8).sum() for x in all_diffs)  # threshold for "changed"

    print(f"Total parameters: {total_params:,}")
    print(f"Changed parameters: {changed_params:,}")
    print(f"Percentage changed: {100 * changed_params / total_params:.2f}%")
    print()

    # Statistics on magnitude of changes
    all_diff_values = jnp.concatenate([x.flatten() for x in all_diffs])
    print(f"Change statistics:")
    print(f"  Mean absolute change: {jnp.mean(all_diff_values):.6f}")
    print(f"  Median absolute change: {jnp.median(all_diff_values):.6f}")
    print(f"  Max absolute change: {jnp.max(all_diff_values):.6f}")
    print(f"  Min absolute change: {jnp.min(all_diff_values):.6f}")

In [ ]:
analyze_state_changes(initial_state, resulting_state)